# In Class Activity April 14th, 2026

In [1]:
# pip install optuna

### Importing libraries, preparing data, initial EDA

In [26]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, roc_auc_score
import optuna

In [27]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [28]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





There is missing data and target class imbalance.  There is also a significant portion of the population (over 20%) who are in the `other` category for education, making it useless (potentially), since it combines doctorates and HS dropouts together.

### Data Preprocessing (minimal) and Baseline Model

In [29]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

C:\Users\jfigg\AppData\Local\Temp\ipykernel_36292\2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [30]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [31]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 
print(f'ROC AUC score: {roc_auc_score(y, xgb_cv.fit(X, y).predict_proba(X)[:, 1])}')

Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588
ROC AUC score: 0.9565623023910909


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

My base model is pretty good when compared to the version found in class, and has an extremely high ROC AUC score.  To improve the model, I will probably just tune it and see what happens.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [32]:
scale_weight = y.value_counts()[0] / y.value_counts()[1]
print(f'Scale pos weight: {scale_weight}')
xgb_param_grid = {
    'scale_pos_weight': [1, 2, 5, 10, scale_weight],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_gscv = GridSearchCV(estimator=XGBClassifier(enable_categorical=True, random_state=42),
                        param_grid=xgb_param_grid,
                        scoring='f1',
                        cv=skf,
                        n_jobs=-1)
xgb_gscv.fit(X_train, y_train)

Scale pos weight: 3.179173440574998


,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'scale_pos_weight': [1, 2, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [33]:
xgb_gscv.best_params_

{'learning_rate': 0.2, 'max_depth': 5, 'scale_pos_weight': 2}

The best model has a high learning rate of .2, a max depth of 5, and a scale of 2 for weighting purposes.

In [34]:
xgb_gscv.best_score_

np.float64(0.7276089235024191)

In [35]:
xgb_gscv.score(X_test, y_test)

0.7309425827944347

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [36]:
scale_weight = y.value_counts()[0] / y.value_counts()[1]
xgb_param_grid = {
    'scale_pos_weight': [1, 2, 5, 10, scale_weight],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'gamma': [0, 1, 5],
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_gscv = GridSearchCV(estimator=XGBClassifier(
                                    enable_categorical=True, 
                                    random_state=42, 
                                    device='cuda'
                                    ),
                        param_grid=xgb_param_grid,
                        scoring='f1',
                        cv=skf,
                        n_jobs=-1)
xgb_gscv.fit(X_train, y_train)

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'gamma': [0, 1, ...], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'scale_pos_weight': [1, 2, ...], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [37]:
xgb_gscv.best_params_

{'gamma': 0,
 'learning_rate': 0.1,
 'max_depth': 7,
 'scale_pos_weight': 2,
 'subsample': 1}

In [38]:
xgb_gscv.best_score_

np.float64(0.7281397250686329)

In [39]:
xgb_gscv.score(X_test, y_test)

0.7291625808982153

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [40]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': [1, 2, 5, 10, scale_weight],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'gamma': [0, 1, 5],
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(enable_categorical=True, random_state=42, device='cuda'),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                subsample=xgb_random.best_params_['subsample'],
                                gamma=xgb_random.best_params_['gamma'],
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'subsample': 0.8, 'scale_pos_weight': 2, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 0}
Best F1 score from RandomizedSearchCV: 0.7190863024387146
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      7431
           1       0.68      0.78      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.82      9769
weighted avg       0.87      0.86      0.86      9769



In [41]:
xgb_random_best.score(X_test, y_test)

0.8598628314054663

I genuinely have no clue why this happened, but the score on the test data is much higher than when I used `GridSearchCV`.  I have investigated but can't find any difference between the models (they use the same parameters), so I will chalk it up as an anomaly and move on.

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [42]:
from catboost import CatBoostClassifier

In [43]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_w = trial.suggest_float('scale_pos_weight', 1, 10)
    max_depth = trial.suggest_int('max_depth', 4, 10)
    learning_rate = trial.suggest_float('learning_rate', .001, .3)
    subsample = trial.suggest_float('subsample', 0.8, 1.0)
    gamma = trial.suggest_float('gamma', 0.0, 10.0)

    model = XGBClassifier(
        random_state=42,
        learning_rate=learning_rate,
        max_depth=max_depth,
        gamma=gamma,
        subsample=subsample,
        scale_pos_weight=scale_pos_w,
        enable_categorical=True
    )

    cv_scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(
        random_state=42,
        learning_rate=study.best_params['learning_rate'],
        max_depth=study.best_params['max_depth'],
        gamma=study.best_params['gamma'],
        subsample=study.best_params['subsample'],
        scale_pos_weight=study.best_params['scale_pos_weight'],
        enable_categorical=True
    )
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-14 21:58:43,104] A new study created in memory with name: no-name-7cabf7e4-da32-47b0-8f52-b6ddd2089651


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-14 21:58:45,045] Trial 0 finished with value: 0.6644500743152438 and parameters: {'scale_pos_weight': 9.217745686192686, 'max_depth': 7, 'learning_rate': 0.09163945463601944, 'subsample': 0.8965984496541188, 'gamma': 3.9280801123857225}. Best is trial 0 with value: 0.6644500743152438.
[I 2026-04-14 21:58:48,282] Trial 1 finished with value: 0.6563369867213528 and parameters: {'scale_pos_weight': 7.672390654048701, 'max_depth': 10, 'learning_rate': 0.01949228688614617, 'subsample': 0.8279310451306231, 'gamma': 7.050269063591922}. Best is trial 0 with value: 0.6644500743152438.
[I 2026-04-14 21:58:49,601] Trial 2 finished with value: 0.6812721494112992 and parameters: {'scale_pos_weight': 6.47271982597821, 'max_depth': 10, 'learning_rate': 0.24533539444586583, 'subsample': 0.9149601387993268, 'gamma': 8.202409183982937}. Best is trial 2 with value: 0.6812721494112992.
[I 2026-04-14 21:58:51,237] Trial 3 finished with value: 0.6658483569142291 and parameters: {'scale_pos_weight

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


I am used to `GridSearchCV`, and I `RandomizedSearchCV` uses the exact same syntax, so I didn't think those methods were difficult.  Regarding `optuna`, I think that it's an interesting approach, and pretty intuitive.  However, it doesn't really fit into the `sklearn` ecosystem, in the same way that a Windows laptop doesn't fit into the Apple ecosystem, despite the windows laptop probably being a better laptop on its own.  As such, it seems that optuna may be better individually, but since the grid searches work with `sklearn` pipelines, I still prefer it.